# Explore the Yelp data (McDonald's and IKEA)

Before a model reads any review, we look at the data: which locations are reviewed, how the ratings are distributed, and who writes reviews.

In this notebook, we will:
1. Import the McDonald's and IKEA locations and compare their ratings.
2. Import all IKEA reviews and explore stars, time, stores and review length.
3. Anonymize review texts and few-shot examples locally and save copies for notebooks 02–05.
4. Read some reviews and look for what a single star rating cannot show.

**Time:** about 15 minutes for exploration, plus a few minutes for local anonymization and inspection

**Before you start: get the data from Moodle**
1. Download `llm_fundamentals_data.zip` from Moodle (Week 10).
2. Drag the zip file into the file Explorer, onto the folder `data`.
3. Run the cells below up to **Unzip the data**: they unpack the zip into `data`.

After unzipping, the folder `data` looks like this:

```
data/
├── businesses_mcdonalds_ikea.csv
├── ikea_reviews.csv
├── fewshot_examples.json
├── TERMS_OF_USE.txt
├── llm_fundamentals_data.zip
├── README.md
├── processed/
└── raw/
```

## Libraries and settings

In [ ]:
# Libraries
from llm_course import DATA, ROOT, anonymize
import os
import json
import platform
import zipfile
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display


# Show current working directory
print(os.getcwd())

## Unzip the data

Unpacks `data/llm_fundamentals_data.zip` from Moodle into `data`. Only needed once.

In [ ]:
# Zip file from Moodle, dragged onto the folder data
ZIP_FILE = DATA / 'llm_fundamentals_data.zip'

if os.path.exists(DATA / 'ikea_reviews.csv'):
    print('Data already unzipped.')
elif os.path.exists(ZIP_FILE):
    with zipfile.ZipFile(ZIP_FILE) as z:
        z.extractall(DATA)
        print('Unzipped into data:', z.namelist())
else:
    print('Zip file not found: download llm_fundamentals_data.zip from Moodle (Week 10),\n'
          'drag it onto the folder data, then run this cell again.')

## Import the business records

In [ ]:
# Read the McDonald's and IKEA locations (data from Moodle, see README: Data)
df_bus = pd.read_csv(
    DATA / 'businesses_mcdonalds_ikea.csv', sep=',', encoding='utf-8'
)

# Shape and first rows
print(df_bus.shape)
df_bus.head(5)

## Number of locations and reviews per chain

In [ ]:
# Locations, reviews and mean rating per chain
summary = df_bus.groupby('chain').agg(
    locations=('business_id', 'count'),
    reviews=('review_count', 'sum'),
    mean_stars=('stars', 'mean'),
)
summary.round(2)

## Average rating: per location vs. per review

Weighting by review count gives larger locations more influence. This approximates a review-level mean: location ratings are already rounded to 0.5 stars, and business-file review counts differ from the review extract. These locations and self-selected reviewers are not representative of all customers.

In [ ]:
# Average of the location ratings, weighted by the number of reviews
df_bus['stars_x_reviews'] = df_bus['stars'] * df_bus['review_count']
grouped = df_bus.groupby('chain')
summary['weighted_stars'] = (
    grouped['stars_x_reviews'].sum() / grouped['review_count'].sum()
)

summary.round(2)

## Distribution of the location ratings

In [ ]:
# One bar chart per chain
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (chain, group) in zip(axes, df_bus.groupby('chain')):
    group['stars'].value_counts().sort_index().plot.bar(ax=ax, color='steelblue')
    ax.set_title(f'{chain}: {len(group)} locations')
    ax.set_xlabel('Stars of the location (rounded to 0.5)')
    ax.set_ylabel('Number of locations')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Locations per state

In [ ]:
# Number of locations per state
pd.crosstab(df_bus['state'], df_bus['chain']).sort_values(
    "McDonald's", ascending=False
)

## The IKEA locations

In [ ]:
# All IKEA records
df_bus.loc[
    df_bus['chain'] == 'IKEA', ['name', 'city', 'state', 'stars', 'review_count']
]

## Import the IKEA reviews

In [ ]:
# Read all IKEA reviews (data from Moodle, see README: Data)
df = pd.read_csv(
    DATA / 'ikea_reviews.csv', sep=',', encoding='utf-8', parse_dates=['date']
)

# Shape and first rows
print(df.shape)
df.head(5)

## Anonymize the texts for notebooks 02–05

Run these cells **once before continuing with the API notebooks**. No API key or OpenAI call is needed. Presidio and the English spaCy model run on this machine and replace detected names, locations, dates, contact details and other identifiers with `<REDACTED>`. The first run can take a few minutes; the progress bar shows how many reviews have been processed.

If the privacy dependencies or model are missing, install them in the selected Python kernel's environment using the commands in the README: `python -m pip install -r requirements.txt -c constraints-tested.txt`, then `python -m spacy download en_core_web_sm`. Restart the kernel after installing. The code stops rather than saving unprocessed text if detection fails.

We prepare a copy of the reviews and the few-shot examples. The original `df` remains available for the local exploration below. Original review IDs and store/date metadata are retained **only for local joins and analysis**: these files are private course working files, not an anonymous dataset suitable for publication. RAG uses local citation aliases when calling OpenAI.


In [ ]:
# Anonymize the review texts locally; keep the original DataFrame unchanged.
preparation_complete = False  # A failed or interrupted run must never save raw copies.
prepared_reviews = df.copy()
prepared_reviews['text'] = [
    anonymize(review).strip()
    for review in tqdm(df['text'], total=len(df), desc='Anonymizing reviews')
]

# Examples can also contain personal information, so prepare them here too.
with open(DATA / 'fewshot_examples.json', encoding='utf-8') as f:
    original_examples = json.load(f)
prepared_examples = []
for example in original_examples:
    prepared = dict(example)
    prepared['text'] = anonymize(example['text']).strip()
    if isinstance(example.get('note'), str):
        prepared['note'] = anonymize(example['note'])
    prepared_examples.append(prepared)

changed = prepared_reviews['text'].ne(df['text'])
print(f'Prepared {len(prepared_reviews):,} reviews and {len(prepared_examples)} examples.')
print(f'Review texts changed: {changed.sum():,}')

preparation_complete = True


### Inspect the result locally

Automated redaction can miss personal details or identifying stories, especially in languages other than English, and can remove useful words. Read the prepared texts locally before saving. The next cell shows ten changed reviews side by side; inspect additional rows as needed using `prepared_reviews`. Correct remaining identifiers in `prepared_reviews['text']` and `prepared_examples` before running the save cell. If sufficient de-identification is not feasible, use synthetic data or local inference.

The preview contains original text and must remain private. Clear notebook outputs before sharing your work. Rerunning the preparation cell starts from the originals again and replaces any manual corrections in memory.


In [ ]:
# Local preview only: original versus prepared text for ten changed reviews.
preview = df.loc[changed, ['review_id', 'text']].head(10).rename(columns={'text': 'original_text'})
preview['prepared_text'] = prepared_reviews.loc[preview.index, 'text']
with pd.option_context('display.max_colwidth', None):
    display(preview)

# Inspect the prepared few-shot examples too.
for i, example in enumerate(prepared_examples, start=1):
    print(f"Example {i} ({example['sentiment']}): {example['text']}")


### Save the prepared files

After inspecting and correcting the texts above, run this cell. Notebooks 02–05 load these saved copies; they give an instruction to run notebook 01 if the files are missing. Saving again replaces the previous prepared copies. The API helpers still repeat local redaction immediately before sending prompt text, examples and history.


In [ ]:
# Save prepared copies locally; never overwrite the original Moodle files.
if not globals().get('preparation_complete', False):
    raise RuntimeError('Complete the local anonymization preparation cell successfully before saving.')
processed = DATA / 'processed'
processed.mkdir(parents=True, exist_ok=True)
reviews_path = processed / 'ikea_reviews_anonymized.csv'
examples_path = processed / 'fewshot_examples_anonymized.json'
prepared_reviews.to_csv(reviews_path, index=False)
examples_path.write_text(json.dumps(prepared_examples, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Saved:', reviews_path)
print('Saved:', examples_path)
print('Ready for notebooks 02–05. These working files remain private and ignored by git.')


## Star distribution of the reviews

In [ ]:
# Number of reviews per star rating
counts = df['stars'].value_counts().sort_index()

ax = counts.plot.bar(color='steelblue', figsize=(6, 4))
ax.bar_label(ax.containers[0])
plt.title(f'All {len(df):,} IKEA reviews by rating')
plt.xlabel('Stars')
plt.ylabel('Number of reviews')
plt.xticks(rotation=0)
plt.show()

Many 4s and 5s, many 1s, few in between: the typical **J-shape** of online reviews.

## Reviews over time

In [ ]:
# Number of reviews and mean rating per year
per_year = df.groupby(df['date'].dt.year).agg(
    reviews=('review_id', 'count'),
    mean_stars=('stars', 'mean'),
)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(per_year.index, per_year['reviews'], color='lightsteelblue')
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of reviews')

ax2 = ax1.twinx()
ax2.plot(per_year.index, per_year['mean_stars'], color='darkred', marker='o')
ax2.set_ylabel('Mean stars', color='darkred')
ax2.set_ylim(1, 5)

plt.title('IKEA reviews per year')
plt.show()

## Ratings by store

In [ ]:
# Reviews, mean rating and time span per store
df.groupby(['store', 'city']).agg(
    reviews=('review_id', 'count'),
    mean_stars=('stars', 'mean'),
    first_review=('date', 'min'),
    last_review=('date', 'max'),
).sort_values('reviews', ascending=False).round(2)

## Review length

In [ ]:
# Number of words per review
df['words'] = df['text'].str.split().str.len()
print(df['words'].describe().round(1))

# Review length by star rating
df.boxplot(column='words', by='stars', showfliers=False, figsize=(6, 4))
plt.suptitle('')
plt.title('Review length by stars (without outliers)')
plt.xlabel('Stars')
plt.ylabel('Words')
plt.show()

## Read some reviews

In [ ]:
# One random review per star rating
for stars, group in df.groupby('stars'):
    row = group.sample(1, random_state=42).iloc[0]
    print(f"{stars} stars | {row['store']}, {row['city']} | {row['date']:%d %b %Y}")
    print(row['text'][:500], '\n')

## Search for a topic

In [ ]:
# TODO: try other topics, e.g. 'assembly', 'return', 'parking', 'staff'
topic = 'meatball'

hits = df[df['text'].str.contains(topic, case=False)]
print(f"{len(hits)} reviews mention '{topic}'")
print(f"Mean stars: {hits['stars'].mean():.2f} (all reviews: {df['stars'].mean():.2f})")
hits[['stars', 'store', 'text']].head(5)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')